# OTA attribution: PFET triode-boundary fine-tune

**Goal:** Verify that the PFET triode-boundary fine-tune reduced M4 IV error and
improved OTA max|ΔV| at L=0.40 µm.

| | Production baseline | Triode fine-tune |
|---|---|---|
| PFET checkpoint | `mosfet_pmos_exp06_sweep_aug_CzBVmMi4.pt` | `pmos_exp07_triode_finetune_KG1HfPbJ.pt` |
| Training data | 44K (no triode aug) | 46K (+ 2K triode, Vsd 0–0.3V) |
| Fine-tune strategy | — | frozen backbone, 50 epochs, LR=1e-4 |

**Probes (both require NGSpice + sky130 PDK):**
- **Probe 1:** Per-device IV error at SPICE node voltages along the transient trajectory
- **Composition metrics:** Pearson r, max|ΔV|, slew accuracy vs SPICE

Run top-to-bottom. Stages 1 and 2 invoke NGSpice; skip them if already run.

## Configuration

In [ ]:
from pathlib import Path
import json

REPO_ROOT = Path("/app")

# OTA geometry — production sizing (matches docs/assets/ota_5t_fno_l040 baseline)
DIFF_W   = 8.0
MIRROR_W = 8.0
TAIL_W   = 2.0
NFET_L   = 0.40
PFET_L   = 0.40
TAIL_L   = 0.40
DEVICE   = "cuda"

# Production baseline (archived to docs/assets)
PROD_COMP_JSON     = REPO_ROOT / "docs/assets/ota_5t_fno_l040/summary.json"
PROD_ATTR_JSON     = REPO_ROOT / "docs/assets/ota_5t_fno_l040/attribution/probe1_summary.json"

# Triode-boundary fine-tune (results in scratch; run dir created by compose_ota)
TRIODE_RUN_DIR     = REPO_ROOT / "scratch/ota_5t_exp07_l040"
TRIODE_COMP_JSON   = TRIODE_RUN_DIR / "summary.json"
TRIODE_ATTR_JSON   = TRIODE_RUN_DIR / "attribution/probe1_summary.json"

# Fine-tune PFET checkpoint + dataset
PFET_CKPT_TRIODE   = REPO_ROOT / "spino/models/mosfet/pfet/pmos_exp07_triode_finetune_KG1HfPbJ.pt"
PFET_DS_TRIODE     = REPO_ROOT / "datasets/sky130_pmos_50k_triode.h5"

print("Production baseline files:")
for p in [PROD_COMP_JSON, PROD_ATTR_JSON]:
    print(f"  {'OK' if p.exists() else 'MISSING':>8}  {p}")
print()
print("Triode fine-tune files:")
for p in [TRIODE_COMP_JSON, TRIODE_ATTR_JSON, PFET_CKPT_TRIODE, PFET_DS_TRIODE]:
    print(f"  {'OK' if p.exists() else 'MISSING':>8}  {p}")

## Stage 1 — Compose OTA with the triode-tuned PFET (skip if already done)

In [ ]:
import subprocess, sys

if TRIODE_COMP_JSON.exists():
    print(f"Triode fine-tune compose output exists — skipping.\n  {TRIODE_COMP_JSON}")
else:
    print("Running compose_ota with triode fine-tune PFET checkpoint...")
    result = subprocess.run(
        [
            sys.executable, "-m", "spino.circuit.compose_ota",
            "--pfet-checkpoint", str(PFET_CKPT_TRIODE),
            "--pfet-dataset",    str(PFET_DS_TRIODE),
            "--pfet-l",          str(PFET_L),
            "--nfet-l",          str(NFET_L),
            "--tail-l",          str(TAIL_L),
            "--diff-w",          str(DIFF_W),
            "--mirror-w",        str(MIRROR_W),
            "--tail-w",          str(TAIL_W),
            "--device",          DEVICE,
            "--output-dir",      str(TRIODE_RUN_DIR),
        ],
        check=False, text=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    )
    print(result.stdout[-4000:] if len(result.stdout) > 4000 else result.stdout)
    if result.returncode != 0:
        raise RuntimeError(f"compose_ota failed (exit {result.returncode})")

s_triode = json.loads(TRIODE_COMP_JSON.read_text())
print(f"\nTriode fine-tune compose: pearson_r={s_triode['transient']['pearson_r']:.4f}  "
      f"max|ΔV|={s_triode['transient']['max_abs_error_v']*1e3:.1f} mV")

## Stage 2 — Run Probe 1 attribution on the fine-tune run (skip if already done)

In [ ]:
if TRIODE_ATTR_JSON.exists():
    print(f"Triode fine-tune attribution exists — skipping.\n  {TRIODE_ATTR_JSON}")
else:
    print("Running ota_attribution for the triode fine-tune run...")
    result = subprocess.run(
        [
            sys.executable, "-m", "spino.circuit.ota_attribution",
            "--run-dir",         str(TRIODE_RUN_DIR),
            "--diff-w",          str(DIFF_W),
            "--mirror-w",        str(MIRROR_W),
            "--tail-w",          str(TAIL_W),
            "--nfet-l",          str(NFET_L),
            "--pfet-l",          str(PFET_L),
            "--tail-l",          str(TAIL_L),
            "--pfet-checkpoint", str(PFET_CKPT_TRIODE),
            "--pfet-dataset",    str(PFET_DS_TRIODE),
            "--device",          DEVICE,
        ],
        check=False, text=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    )
    print(result.stdout[-4000:] if len(result.stdout) > 4000 else result.stdout)
    if result.returncode != 0:
        raise RuntimeError(f"ota_attribution failed (exit {result.returncode})")

print("Attribution output:", TRIODE_ATTR_JSON)

## Stage 3 — Per-device IV error: production vs triode fine-tune

In [ ]:
a_prod   = json.loads(PROD_ATTR_JSON.read_text())
a_triode = json.loads(TRIODE_ATTR_JSON.read_text())

devices = [
    ("M1", "M1_nfet_diffpair"),
    ("M2", "M2_nfet_diffpair"),
    ("M3", "M3_pfet_mirror_diode"),
    ("M4", "M4_pfet_mirror_out"),
    ("M5", "M5_nfet_tail"),
]

print(f"{'Device':<6} {'Production max |ΔI|':>22} {'Fine-tune max |ΔI|':>22} {'Reduction':>12}  Notes")
print("-" * 82)
for label, key in devices:
    e_prod   = a_prod[key]["max_abs_error_a"] * 1e6
    e_triode = a_triode[key]["max_abs_error_a"] * 1e6
    reduction = (e_prod - e_triode) / (e_prod + 1e-30)
    note = "  <-- triode target" if label == "M4" else ""
    print(f"{label:<6} {e_prod:>19.2f} µA  {e_triode:>19.2f} µA  {reduction:>10.1%}{note}")

## Stage 4 — Per-device bar chart

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

labels_bar  = [d[0] for d in devices]
vals_prod   = [a_prod[k]["max_abs_error_a"]   * 1e6 for _, k in devices]
vals_triode = [a_triode[k]["max_abs_error_a"] * 1e6 for _, k in devices]

x = np.arange(len(labels_bar))
w = 0.35

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(x - w/2, vals_prod,   w, label="Production",        color="#cc6600", alpha=0.85)
ax.bar(x + w/2, vals_triode, w, label="Triode fine-tune",  color="#0066cc", alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(labels_bar, fontsize=12)
ax.set_ylabel("Peak |ΔI| (µA)")
ax.set_title("Per-device peak IV error: production vs triode fine-tune (L=0.40 µm)")
ax.legend()
ax.grid(True, alpha=0.3, axis="y")

for xi, (v_p, v_t) in zip(x, zip(vals_prod, vals_triode)):
    ax.text(xi - w/2, v_p + 0.15, f"{v_p:.1f}", ha="center", fontsize=8, color="#883300")
    ax.text(xi + w/2, v_t + 0.15, f"{v_t:.1f}", ha="center", fontsize=8, color="#004499")

plt.tight_layout()
out_fig = TRIODE_RUN_DIR / "attribution" / "device_errors_comparison.png"
out_fig.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(out_fig, dpi=150)
plt.show()
print("Saved:", out_fig)

## Stage 5 — Composition metrics: production vs triode fine-tune

In [ ]:
c_prod   = json.loads(PROD_COMP_JSON.read_text())
c_triode = json.loads(TRIODE_COMP_JSON.read_text())

def tran(s):
    t = s["transient"]
    return dict(
        pearson_r    = t["pearson_r"],
        max_dv_mv    = t["max_abs_error_v"] * 1e3,
        slew_v_us    = t["fno"]["slew_rate_v_per_us"],
        slew_err_pct = t["slew_rate_rel_err"] * 100,
    )

m_prod   = tran(c_prod)
m_triode = tran(c_triode)

print(f"{'Metric':<24} {'Production':>12} {'Fine-tune':>12}  Gate")
print("-" * 62)
print(f"{'Pearson r':<24} {m_prod['pearson_r']:>12.4f} {m_triode['pearson_r']:>12.4f}  > 0.997")
g_prod   = 'PASS' if m_prod['max_dv_mv']   <= 30 else 'FAIL'
g_triode = 'PASS' if m_triode['max_dv_mv'] <= 30 else f'NEAR ({m_triode["max_dv_mv"]-30:+.1f} mV)'
print(f"{'max|ΔV| (mV)':<24} {m_prod['max_dv_mv']:>12.1f} {m_triode['max_dv_mv']:>12.1f}  <= 30 mV   {g_prod} → {g_triode}")
print(f"{'FNO slew (V/µs)':<24} {m_prod['slew_v_us']:>12.2f} {m_triode['slew_v_us']:>12.2f}")
print(f"{'Slew err (%)':<24} {m_prod['slew_err_pct']:>12.2f} {m_triode['slew_err_pct']:>12.2f}")
print()
print(f"max|ΔV| improvement: {m_prod['max_dv_mv']:.1f} → {m_triode['max_dv_mv']:.1f} mV  "
      f"({(m_prod['max_dv_mv']-m_triode['max_dv_mv'])/m_prod['max_dv_mv']:.0%} reduction)")